# Kalshi Weather Markets — Historical Backtester

Covers **5 cities**: Los Angeles · New York · Chicago · Miami · Denver

**Methodology**
- Fetch all finalized Kalshi weather markets + tick data via `pykalshi`.
- Aggregate ticks to **hourly VWAP** prices (6 am – 11 am local only).
- Load matching **hourly weather** data via `meteostat`.
- Train a per-city **Random Forest Regressor** on morning weather → daily high.
- Convert point prediction + residual uncertainty to **bracket probabilities** (Gaussian CDF).
- Compute `edge = model_prob − market_implied_prob`. Signal when `|edge| ≥ 8%`.
- Apply same filters as live system: noon cutoff, market prob ≥ 3%, model prob < 100%.
- Simulate **limit-order** entries with three slippage scenarios.
- Deduct realistic **maker fees** from every trade.


## Cell 1 — Imports & Configuration

In [ ]:
import os, re, time, warnings, logging
from pathlib import Path
from datetime import datetime, date, timedelta
from typing import Optional
from zoneinfo import ZoneInfo

import numpy as np
import pandas as pd
from scipy.stats import norm as scipy_norm
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_absolute_error
import meteostat as ms
from dotenv import load_dotenv

warnings.filterwarnings('ignore')
logging.basicConfig(level=logging.INFO,
                    format='%(asctime)s  %(levelname)-8s  %(message)s',
                    datefmt='%H:%M:%S')
logger = logging.getLogger('backtest')
load_dotenv()

# ── City config ────────────────────────────────────────────────────────────
CITIES = {
    'Los Angeles': {'lat': 34.0522, 'lon': -118.2437, 'ticker_prefix': 'KXHIGHLAX', 'timezone': 'America/Los_Angeles'},
    'New York':    {'lat': 40.7128, 'lon': -74.0060,  'ticker_prefix': 'KXHIGHNY',  'timezone': 'America/New_York'},
    'Chicago':     {'lat': 41.8781, 'lon': -87.6298,  'ticker_prefix': 'KXHIGHCHI', 'timezone': 'America/Chicago'},
    'Miami':       {'lat': 25.7617, 'lon': -80.1918,  'ticker_prefix': 'KXHIGHMIA', 'timezone': 'America/New_York'},
    'Denver':      {'lat': 39.7392, 'lon': -104.9903, 'ticker_prefix': 'KXHIGHDEN', 'timezone': 'America/Denver'},
}

# ── Strategy parameters ────────────────────────────────────────────────────
TRADE_HOUR_START  = 6     # earliest local hour to consider a signal
TRADE_HOUR_CUTOFF = 12    # no trades at or after noon local
MIN_EDGE          = 0.15  # minimum |model_prob - market_prob|
MIN_MARKET_PROB   = 0.03  # skip brackets priced below 3 cents
MAX_MODEL_PROB    = 1.00  # skip 100%-certain model predictions (exclusive)
CONTRACTS         = 10    # contracts per trade

# ── Fee model  (Kalshi limit-order / maker fee) ────────────────────────────
# Kalshi charges approximately $0.007 per $1 face-value contract for maker fills.
# Applied on entry only (exit is settlement at $1.00 or $0.00 — no exit fee).
MAKER_FEE_PER_CONTRACT = 0.007

# ── Slippage scenarios (extra cost per contract in dollars) ───────────────
# No slippage   : fill at hourly VWAP (best case — our limit resting at the bid)
# Moderate      : VWAP + $0.01 (we shade our limit up 1 cent for faster fill)
# High          : VWAP + $0.03 (aggressive limit, close to ask)
SLIPPAGE_SCENARIOS = {
    'No Slippage':       0.00,
    'Moderate Slippage': 0.01,
    'High Slippage':     0.03,
}

# ── Data cache directory ──────────────────────────────────────────────────
DATA_DIR = Path('backtest_data')
DATA_DIR.mkdir(exist_ok=True)

HIST_START = datetime(2025, 1, 1)
HIST_END   = datetime.today()

print('Configuration ready.')
for city, cfg in CITIES.items():
    print(f"  {city:<15} {cfg['ticker_prefix']}  ({cfg['timezone']})")


## Cell 2 — Load Historical Kalshi Tick Data (all 5 cities)

In [ ]:
from pykalshi import KalshiClient, MarketStatus


def _make_client() -> KalshiClient:
    return KalshiClient(
        api_key_id=os.getenv('KALSHI_API_KEY_ID'),
        private_key_path=os.getenv('KALSHI_PRIVATE_KEY_PATH'),
    )


def load_city_kalshi_data(
    city_name: str,
    ticker_prefix: str,
    client: KalshiClient,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    Fetch all finalized Kalshi markets + their full tick histories for one city.
    Results are cached to CSV so the API is only called once.

    Returns
    -------
    trades_df  : raw tick data with columns ticker, yes_price, count, created_time
    markets_df : one row per market with ticker, result, settlement_value
    """
    trades_cache  = DATA_DIR / f'{ticker_prefix}_trades.csv'
    markets_cache = DATA_DIR / f'{ticker_prefix}_markets.csv'

    if trades_cache.exists() and markets_cache.exists():
        logger.info('[%s] Loading cached Kalshi data...', city_name)
        trades_df  = pd.read_csv(trades_cache, parse_dates=['created_time'])
        markets_df = pd.read_csv(markets_cache)
        logger.info('[%s] %d trades, %d markets loaded from cache.',
                    city_name, len(trades_df), len(markets_df))
        return trades_df, markets_df

    logger.info('[%s] Fetching all markets from Kalshi API...', city_name)
    all_markets = client.get_markets(series_ticker=ticker_prefix, fetch_all=True)

    # Only finalized markets have a known outcome
    finalized = [m for m in all_markets if m.data.status == 'finalized']
    logger.info('[%s] %d finalized markets found.', city_name, len(finalized))

    # Build a compact markets summary
    mkt_rows = []
    for m in finalized:
        try:
            settle_raw = m.data.settlement_value_dollars
            settle = float(settle_raw) if settle_raw is not None else None
        except (TypeError, ValueError):
            settle = None
        mkt_rows.append({
            'ticker':           m.ticker,
            'result':           m.data.result,
            'settlement_value': settle,   # $1.00 = YES won, $0.00 = NO won
            'close_time':       m.data.close_time,
        })
    markets_df = pd.DataFrame(mkt_rows)
    markets_df.to_csv(markets_cache, index=False)

    # Fetch tick data for every finalized market
    all_trades: list[pd.DataFrame] = []
    for i, mkt in enumerate(finalized):
        try:
            ticks = client.get_trades(ticker=mkt.ticker, fetch_all=True)
            if ticks:
                df_t = ticks.to_dataframe()
                df_t['ticker'] = mkt.ticker
                all_trades.append(df_t)
        except Exception as exc:
            logger.warning('[%s] %s: trade fetch failed — %s', city_name, mkt.ticker, exc)
        if (i + 1) % 100 == 0:
            logger.info('[%s] %d / %d markets processed...', city_name, i + 1, len(finalized))
        time.sleep(0.05)   # gentle rate-limiting

    if not all_trades:
        logger.error('[%s] No trade data retrieved!', city_name)
        return pd.DataFrame(), markets_df

    trades_df = pd.concat(all_trades, ignore_index=True)
    trades_df['created_time'] = pd.to_datetime(trades_df['created_time'], utc=True)
    trades_df.to_csv(trades_cache, index=False)
    logger.info('[%s] Saved %d trades to cache.', city_name, len(trades_df))
    return trades_df, markets_df


# ── Run for all 5 cities ──────────────────────────────────────────────────
client = _make_client()
city_trades:  dict[str, pd.DataFrame] = {}
city_markets: dict[str, pd.DataFrame] = {}

for city_name, cfg in CITIES.items():
    trades_df, markets_df = load_city_kalshi_data(city_name, cfg['ticker_prefix'], client)
    city_trades[city_name]  = trades_df
    city_markets[city_name] = markets_df
    print(f"  {city_name:<15} {len(trades_df):>10,} trades   {len(markets_df):>5} markets")


## Cell 3 — Load Historical Weather Data (meteostat, all 5 cities)

In [ ]:
import numpy as np


def load_city_weather(city_name: str, lat: float, lon: float, timezone: str) -> pd.DataFrame:
    """
    Fetch hourly weather from meteostat for a city, convert to local time,
    apply the same feature engineering used in Data_Sorting.ipynb.

    Columns returned (all on local clock):
        local_date, hour, temp (°F), rhum, wspd, pres, cldc,
        wdir_sin, wdir_cos, condition_* dummies
    """
    cache_file = DATA_DIR / f'{city_name.replace(" ", "_")}_weather.csv'
    if cache_file.exists():
        logger.info('[%s] Loading cached weather...', city_name)
        df = pd.read_csv(cache_file, parse_dates=['time'])
        df['local_date'] = pd.to_datetime(df['local_date']).dt.date
        return df

    logger.info('[%s] Fetching meteostat hourly weather...', city_name)
    point    = ms.Point(lat, lon)
    stations = ms.stations.nearby(point, limit=1)
    hourly   = ms.hourly(stations, HIST_START, HIST_END)
    raw      = hourly.fetch().reset_index()

    # Convert UTC → local time
    raw['time'] = raw['time'].dt.tz_localize('UTC').dt.tz_convert(timezone)
    raw['local_date'] = raw['time'].dt.date
    raw['hour']       = raw['time'].dt.hour

    # Fahrenheit conversion
    raw['temp'] = raw['temp'] * 9 / 5 + 32

    # Cyclical wind direction
    raw['wdir_rad'] = np.deg2rad(raw['wdir'].fillna(0))
    raw['wdir_sin'] = np.sin(raw['wdir_rad'])
    raw['wdir_cos'] = np.cos(raw['wdir_rad'])
    raw = raw.drop(columns=['wdir', 'wdir_rad'], errors='ignore')

    # Weather condition dummies
    if 'coco' in raw.columns:
        raw = pd.get_dummies(raw, columns=['coco'], prefix='condition', dtype=int)

    # Drop unused columns
    raw = raw.drop(columns=['station', 'prcp', 'snwd', 'tsun', 'wpgt'], errors='ignore')

    raw.to_csv(cache_file, index=False)
    logger.info('[%s] Saved %d hourly weather rows.', city_name, len(raw))
    return raw


city_weather: dict[str, pd.DataFrame] = {}
for city_name, cfg in CITIES.items():
    df_w = load_city_weather(city_name, cfg['lat'], cfg['lon'], cfg['timezone'])
    city_weather[city_name] = df_w
    print(f"  {city_name:<15} {len(df_w):>8,} hourly rows  "
          f"({df_w['local_date'].min()} → {df_w['local_date'].max()})")


## Cell 4 — Preprocess: Merge & Build Feature Matrix

In [ ]:
_BRACKET_RE = re.compile(r'-\d{2}[A-Z]{3}\d{2}-(.+)$', re.IGNORECASE)


def extract_bracket_label(ticker: str) -> Optional[str]:
    m = _BRACKET_RE.search(ticker)
    return m.group(1).upper() if m else None


def parse_bracket_label(label: str, b_min_low: float = None, b_max_high: float = None) -> Optional[dict]:
    """Convert a Kalshi bracket label to a bracket-definition dict.

    T-brackets are tail overflow buckets (NOT directional bets):
      Lower T: label == b_min_low        → settles YES if temp < b_min_low
      Upper T: label == b_max_high - 1   → settles YES if temp >= b_max_high
    Context parameters b_min_low / b_max_high are required for correct T classification.
    """
    label = label.upper().strip()
    try:
        if label.startswith('T'):
            val = float(label[1:])
            if b_min_low is not None and b_max_high is not None:
                dist_lower = abs(val - b_min_low)
                dist_upper = abs(val - (b_max_high - 1))
                if dist_lower <= dist_upper:
                    # Lower tail: P(temp < b_min_low)
                    return {'label': label, 'high': b_min_low, 'type': 'below'}
                else:
                    # Upper tail: P(temp >= b_max_high)
                    return {'label': label, 'low': b_max_high, 'type': 'above'}
            else:
                # No context — skip (return None so this row is filtered out)
                return None
        elif label.startswith('B'):
            val = float(label[1:])
            if '.' in label[1:]:
                low = val - 0.5
                return {'label': label, 'low': low, 'high': low + 2.0, 'type': 'between'}
            else:
                return {'label': label, 'high': val, 'type': 'below'}
    except (ValueError, TypeError):
        pass
    return None


def build_city_dataset(city_name: str, timezone: str) -> pd.DataFrame:
    """
    For one city, merge:
      - Hourly Kalshi VWAP prices (morning only: TRADE_HOUR_START to TRADE_HOUR_CUTOFF-1)
      - Hourly weather features at those same hours
      - The actual daily high temperature (ground truth for the model)
      - Whether YES won (ground truth for settlement)

    Returns one row per (date, bracket, hour) triple.
    """
    trades_df  = city_trades.get(city_name, pd.DataFrame())
    markets_df = city_markets.get(city_name, pd.DataFrame())
    weather_df = city_weather.get(city_name, pd.DataFrame())

    if trades_df.empty or markets_df.empty or weather_df.empty:
        logger.warning('[%s] Missing data — skipping.', city_name)
        return pd.DataFrame()

    # ── Step A: Attach bracket label to each trade ──
    trades_df = trades_df.copy()
    trades_df['bracket_label'] = trades_df['ticker'].apply(extract_bracket_label)
    trades_df.dropna(subset=['bracket_label'], inplace=True)

    # Convert to local time for hour/date extraction
    trades_df['created_time'] = pd.to_datetime(trades_df['created_time'], utc=True)
    trades_df['local_dt']   = trades_df['created_time'].dt.tz_convert(timezone)
    trades_df['local_date'] = trades_df['local_dt'].dt.date
    trades_df['hour']       = trades_df['local_dt'].dt.hour

    # Keep only morning trading window
    trades_df = trades_df[
        (trades_df['hour'] >= TRADE_HOUR_START) &
        (trades_df['hour'] <  TRADE_HOUR_CUTOFF)
    ]

    # ── Step B: Aggregate ticks → hourly VWAP per (ticker, date, hour) ──
    # BUG FIX 1: API returns 'yes_price_dollars' (already in [0,1]) and 'count_fp'
    #            — NOT 'yes_price' and 'count' which don't exist.
    trades_df['yes_price'] = pd.to_numeric(trades_df['yes_price_dollars'], errors='coerce')
    trades_df['count']     = pd.to_numeric(trades_df['count_fp'],          errors='coerce')
    trades_df = trades_df.dropna(subset=['yes_price', 'count'])
    trades_df['notional'] = trades_df['yes_price'] * trades_df['count']

    hourly = (
        trades_df
        .groupby(['ticker', 'bracket_label', 'local_date', 'hour'])
        .agg(
            vwap     =('notional', 'sum'),
            total_vol=('count',    'sum'),
        )
        .reset_index()
    )
    hourly['vwap'] = hourly['vwap'] / hourly['total_vol']

    # BUG FIX 2: yes_price_dollars is already in dollar terms [0,1].
    #            Dividing by 100 would turn a 50% market into 0.5%, making
    #            every bracket look massively mispriced. No division needed.
    hourly['market_implied_prob'] = hourly['vwap']

    # ── Step C: Merge settlement outcomes ──
    markets_df['yes_won'] = (markets_df['settlement_value'].astype(float) >= 0.99).astype(int)
    hourly = hourly.merge(markets_df[['ticker', 'yes_won']], on='ticker', how='left')

    # ── Step D: Compute actual daily high from weather ──
    weather_df = weather_df.copy()
    weather_df['local_date'] = pd.to_datetime(weather_df['local_date']).dt.date
    daily_highs = weather_df.groupby('local_date')['temp'].max().reset_index()
    daily_highs.rename(columns={'temp': 'actual_daily_high'}, inplace=True)

    hourly = hourly.merge(daily_highs, on='local_date', how='left')

    # ── Step E: Merge weather features at each morning hour ──
    weather_morning = weather_df[
        (weather_df['hour'] >= TRADE_HOUR_START) &
        (weather_df['hour'] <  TRADE_HOUR_CUTOFF)
    ].copy()

    hourly = hourly.merge(
        weather_morning.drop(columns=['time'], errors='ignore'),
        on=['local_date', 'hour'],
        how='left',
    )

    # ── Step F: Compute B-bracket range per market (needed to correctly classify T-brackets) ──
    hourly['market_base'] = hourly['ticker'].str.extract(r'^(.*)-[BT][\d.]+$')[0]
    _b_mask = hourly['bracket_label'].str.startswith('B')
    _b_df = hourly.loc[_b_mask, ['market_base', 'bracket_label']].drop_duplicates().copy()
    _b_df['_b_low'] = _b_df['bracket_label'].str[1:].astype(float) - 0.5
    _b_bounds = _b_df.groupby('market_base').agg(
        market_b_min_low=('_b_low', 'min'),
        market_b_max_high=('_b_low', lambda x: x.max() + 2.0),
    ).reset_index()
    hourly = hourly.merge(_b_bounds, on='market_base', how='left')

    hourly['city'] = city_name
    return hourly


city_datasets: dict[str, pd.DataFrame] = {}
for city_name, cfg in CITIES.items():
    ds = build_city_dataset(city_name, cfg['timezone'])
    city_datasets[city_name] = ds
    if not ds.empty:
        print(f"  {city_name:<15} {len(ds):>8,} rows  OK"
              f"  (market_implied_prob range: {ds['market_implied_prob'].min():.3f}–{ds['market_implied_prob'].max():.3f})")
    else:
        print(f"  {city_name:<15}      EMPTY")


## Cell 5 — Train Per-City RF on First 80% of Dates (80/20 split)

One Random Forest is trained per city on the **first 80% of unique dates** in that
city's dataset. No predictions are made on training data. The backtest (Cell 7)
runs only on the **last 20% of dates**.

**Uncertainty** is estimated via `TimeSeriesSplit` OOF on training data only: residuals
from each fold are pooled, and their std dev is the Gaussian spread in the bracket
probability engine (Cell 6).

Trains exactly **5 models** total — runs in ~30 seconds.

In [ ]:
WEATHER_FEATURES = [
    'hour', 'temp', 'rhum', 'wspd', 'pres', 'cldc',
    'wdir_sin', 'wdir_cos',
]

N_OOF_SPLITS = 5


def get_feature_cols(df: pd.DataFrame) -> list[str]:
    cond_cols = [c for c in df.columns if c.startswith('condition_')]
    return [f for f in WEATHER_FEATURES + cond_cols if f in df.columns]


def train_city_model(city_name: str, df: pd.DataFrame) -> dict:
    """
    Train one RF per city on first 80% of unique dates.
    OOF residuals from TimeSeriesSplit on training data estimate uncertainty.
    Returns a dict: model, uncertainty, feat_cols, test_dates, train_dates.
    """
    target    = 'actual_daily_high'
    feat_cols = get_feature_cols(df)

    clean = df.dropna(subset=feat_cols + [target]).copy()
    clean = clean.sort_values(['local_date', 'hour']).reset_index(drop=True)

    unique_dates = sorted(clean['local_date'].unique())
    n_dates      = len(unique_dates)
    split_idx    = int(n_dates * 0.8)

    train_dates     = set(unique_dates[:split_idx])
    test_dates      = set(unique_dates[split_idx:])
    train_date_list = unique_dates[:split_idx]
    test_date_list  = unique_dates[split_idx:]

    print(f"  {city_name:<15}  {n_dates} total dates")
    print(f"    Train: {train_date_list[0]}  →  {train_date_list[-1]}  ({len(train_dates)} dates)")
    print(f"    Test:  {test_date_list[0]}  →  {test_date_list[-1]}  ({len(test_dates)} dates)")

    train_mask = clean['local_date'].isin(train_dates)
    X_train    = clean.loc[train_mask, feat_cols].values
    y_train    = clean.loc[train_mask, target].values

    if len(X_train) == 0:
        logger.error('[%s] No training rows — check city_datasets.', city_name)
        return {}

    # OOF uncertainty on training data only
    tscv       = TimeSeriesSplit(n_splits=N_OOF_SPLITS)
    oof_resids = []
    for tr_idx, val_idx in tscv.split(X_train):
        X_tr, X_val = X_train[tr_idx], X_train[val_idx]
        y_tr, y_val = y_train[tr_idx], y_train[val_idx]
        if len(X_tr) == 0 or len(X_val) == 0:
            continue
        m_fold = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
        m_fold.fit(X_tr, y_tr)
        oof_resids.extend((y_val - m_fold.predict(X_val)).tolist())

    uncertainty = max(float(np.std(oof_resids)), 0.5) if oof_resids else 4.0

    # Final model on all training data
    model = RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1)
    model.fit(X_train, y_train)
    train_mae = mean_absolute_error(y_train, model.predict(X_train))

    print(f"    OOF residual std (uncertainty): {uncertainty:.2f}°F")
    print(f"    Train MAE (in-sample):          {train_mae:.2f}°F")

    return {
        'model':       model,
        'uncertainty': uncertainty,
        'feat_cols':   feat_cols,
        'test_dates':  test_dates,
        'train_dates': train_dates,
    }


print('Training per-city RF models (80/20 date split)...\n')
city_models: dict[str, dict] = {}

for city_name in list(city_datasets.keys()):
    ds = city_datasets[city_name]
    if ds.empty:
        logger.warning('[%s] Empty dataset — skipping.', city_name)
        continue
    result = train_city_model(city_name, ds)
    if result:
        city_models[city_name] = result
        print()

print(f'Done. {len(city_models)} city models trained.')

## Cell 6 — Bracket Probability Engine (Gaussian CDF)

In [ ]:
def bracket_prob_gaussian(predicted_high: float, uncertainty: float, bracket: dict) -> float:
    """
    Compute P(daily_high falls in bracket) assuming
    daily_high ~ N(predicted_high, uncertainty²).

    Parameters
    ----------
    predicted_high : float   RF regressor point estimate (°F)
    uncertainty    : float   std dev of OOF residuals (°F)
    bracket        : dict    from parse_bracket_label()
    """
    if uncertainty <= 0:
        uncertainty = 1e-6

    btype = bracket['type']
    if btype == 'above':
        return float(1.0 - scipy_norm.cdf(bracket['low'], predicted_high, uncertainty))
    elif btype == 'below':
        return float(scipy_norm.cdf(bracket['high'], predicted_high, uncertainty))
    elif btype == 'between':
        return float(
            scipy_norm.cdf(bracket['high'], predicted_high, uncertainty) -
            scipy_norm.cdf(bracket['low'],  predicted_high, uncertainty)
        )
    return 0.0


# ── Quick sanity check ───────────────────────────────────────────────────
# Upper-tail T73 in a market with B-range 66–74: P(above 74) with pred=76 should be > 0.5
_b_upper = parse_bracket_label('T73', b_min_low=66.0, b_max_high=74.0)
assert _b_upper is not None and _b_upper['type'] == 'above', 'Upper-tail T-bracket parse failed'
assert bracket_prob_gaussian(76.0, 3.0, _b_upper) > 0.5, 'Upper-tail probability sanity check failed'

# Lower-tail T66 in same market: P(below 66) with pred=60 should be > 0.5
_b_lower = parse_bracket_label('T66', b_min_low=66.0, b_max_high=74.0)
assert _b_lower is not None and _b_lower['type'] == 'below', 'Lower-tail T-bracket parse failed'
assert bracket_prob_gaussian(60.0, 3.0, _b_lower) > 0.5, 'Lower-tail probability sanity check failed'

# B-bracket between check
_b2 = parse_bracket_label('B72.5')
assert 0 < bracket_prob_gaussian(74.0, 3.0, _b2) < 1, 'Between-bracket sanity check failed'

print('Bracket probability engine: OK')

## Cell 7 — Backtest Engine

In [ ]:
def run_backtest(slippage_per_contract: float) -> pd.DataFrame:
    """
    Backtest on the last 20% of dates (test period) only.
    ONE trade per city per day — the bracket with the highest |edge|.
    Prints a filter funnel per city.
    """
    trade_log: list[dict] = []

    for city_name, ds in city_datasets.items():
        if ds.empty:
            continue
        if city_name not in city_models:
            logger.warning('[%s] No model — run Cell 5 first.', city_name)
            continue

        cm          = city_models[city_name]
        model       = cm['model']
        uncertainty = cm['uncertainty']
        feat_cols   = cm['feat_cols']
        test_dates  = cm['test_dates']

        ds_sorted = ds.sort_values(['local_date', 'bracket_label', 'hour'])

        cnt_total = cnt_test = cnt_hour = cnt_mkt = cnt_brk = cnt_sig = cnt_trade = 0
        date_candidates: dict = {}   # local_date → best candidate dict so far

        for _, row in ds_sorted.iterrows():
            cnt_total += 1

            # Gate 1: test period only
            if row['local_date'] not in test_dates:
                continue
            cnt_test += 1

            # Gate 2: noon cutoff
            if row['hour'] >= TRADE_HOUR_CUTOFF:
                continue
            cnt_hour += 1

            # Gate 3: market prob floor
            market_prob = row.get('market_implied_prob', np.nan)
            if pd.isna(market_prob) or market_prob < MIN_MARKET_PROB:
                continue
            cnt_mkt += 1

            # Gate 4: parse bracket (pass B-range context for correct T-bracket classification)
            b_min_low  = row.get('market_b_min_low',  np.nan)
            b_max_high = row.get('market_b_max_high', np.nan)
            bracket = parse_bracket_label(
                row['bracket_label'],
                None if pd.isna(b_min_low)  else float(b_min_low),
                None if pd.isna(b_max_high) else float(b_max_high),
            )
            if bracket is None:
                continue
            cnt_brk += 1

            # Gate 5: build feature row, skip if any NaN
            row_features = []
            ok = True
            for col in feat_cols:
                val = row.get(col, np.nan)
                if pd.isna(val):
                    ok = False
                    break
                row_features.append(float(val))
            if not ok:
                continue

            predicted_high = float(model.predict(np.array(row_features).reshape(1, -1))[0])
            model_prob     = bracket_prob_gaussian(predicted_high, uncertainty, bracket)

            # Gate 6: model prob ceiling
            if model_prob >= MAX_MODEL_PROB:
                continue

            # Gate 7: edge threshold
            edge = model_prob - market_prob
            if abs(edge) < MIN_EDGE:
                continue
            cnt_sig += 1

            # Keep only the highest |edge| signal per day (one trade per city per day)
            local_date = row['local_date']
            if local_date not in date_candidates or abs(edge) > abs(date_candidates[local_date]['edge']):
                date_candidates[local_date] = {
                    'row':            row,
                    'edge':           edge,
                    'model_prob':     model_prob,
                    'market_prob':    market_prob,
                    'predicted_high': predicted_high,
                }

        # Execute the single best signal per day
        for local_date, cand in sorted(date_candidates.items()):
            row            = cand['row']
            edge           = cand['edge']
            model_prob     = cand['model_prob']
            market_prob    = cand['market_prob']
            predicted_high = cand['predicted_high']

            yes_won = row.get('yes_won', np.nan)
            if pd.isna(yes_won):
                continue

            side = 'YES' if edge > 0 else 'NO'
            vwap = row['market_implied_prob']
            entry_price = min(vwap + slippage_per_contract, 0.99) if side == 'YES' \
                          else min((1.0 - vwap) + slippage_per_contract, 0.99)
            fee_total = MAKER_FEE_PER_CONTRACT * CONTRACTS

            yes_won = int(yes_won)
            won     = (side == 'YES' and yes_won == 1) or (side == 'NO' and yes_won == 0)
            payout  = 1.00 * CONTRACTS if won else 0.00
            net_pnl = (payout - entry_price * CONTRACTS) - fee_total

            cnt_trade += 1
            trade_log.append({
                'city':           city_name,
                'local_date':     local_date,
                'hour':           int(row['hour']),
                'ticker':         row['ticker'],
                'bracket_label':  row['bracket_label'],
                'side':           side,
                'model_prob':     round(model_prob, 4),
                'market_prob':    round(market_prob, 4),
                'edge':           round(edge, 4),
                'predicted_high': round(predicted_high, 2),
                'actual_high':    round(row.get('actual_daily_high', np.nan), 2),
                'entry_price':    round(entry_price, 4),
                'slippage':       round(slippage_per_contract, 4),
                'contracts':      CONTRACTS,
                'fee_total':      round(fee_total, 4),
                'gross_pnl':      round((payout - entry_price * CONTRACTS), 4),
                'net_pnl':        round(net_pnl, 4),
                'won':            int(won),
            })

        print(f"\n[{city_name}] Filter funnel (slippage=${slippage_per_contract:.2f}):")
        print(f"  Total rows:          {cnt_total:>7,}")
        print(f"  In test period:      {cnt_test:>7,}  ({cnt_test/max(cnt_total,1):.1%})")
        print(f"  Pass noon cutoff:    {cnt_hour:>7,}")
        print(f"  Pass mkt_prob>=3%:   {cnt_mkt:>7,}")
        print(f"  Bracket parsed:      {cnt_brk:>7,}")
        print(f"  |edge|>={MIN_EDGE:.0%}:        {cnt_sig:>7,}")
        print(f"  Candidate dates:     {len(date_candidates):>7,}")
        print(f"  Trades placed:       {cnt_trade:>7,}")

    if not trade_log:
        print('\nNo trades generated.')
        return pd.DataFrame()

    df = pd.DataFrame(trade_log).sort_values(['local_date', 'city']).reset_index(drop=True)
    df['cumulative_pnl'] = df['net_pnl'].cumsum()
    return df


print('Backtest engine ready.')


In [ ]:
# === DIAGNOSTIC: check city_models and date alignment ===
print("=== city_models ===")
print(f"Keys: {list(city_models.keys())}")
for city, cm in city_models.items():
    td = cm.get('test_dates', set())
    print(f"  {city}: {len(td)} test dates, example: {sorted(td)[:2]}")

print("\n=== city_datasets ===")
for city, ds in city_datasets.items():
    if ds.empty:
        print(f"  {city}: EMPTY"); continue
    td = city_models.get(city, {}).get('test_dates', set())
    test_rows = ds[ds['local_date'].isin(td)]
    print(f"  {city}: {len(ds)} rows, {len(test_rows)} in test period")
    print(f"    local_date dtype={ds['local_date'].dtype}, type={type(ds['local_date'].iloc[0]).__name__}")
    if test_rows.empty and td:
        print(f"    ds sample dates : {sorted(ds['local_date'].unique())[:3]}")
        print(f"    test_dates sample: {sorted(td)[:3]}")

## Cell 8 — Run All 3 Slippage Scenarios

In [ ]:
backtest_results: dict[str, pd.DataFrame] = {}

for scenario_name, slippage in SLIPPAGE_SCENARIOS.items():
    print(f'\nRunning: {scenario_name}  (slippage = ${slippage:.2f}/contract) ...')
    bt = run_backtest(slippage_per_contract=slippage)
    backtest_results[scenario_name] = bt
    if not bt.empty:
        print(f'  Trades: {len(bt)}   Win rate: {bt["won"].mean():.1%}'
              f'   Net P&L: ${bt["net_pnl"].sum():.2f}'
              f'   Total fees: ${bt["fee_total"].sum():.2f}')
    else:
        print('  No trades.')


## Cell 9 — Results: Scenario Comparison Table

In [ ]:
def scenario_summary(name: str, df: pd.DataFrame) -> dict:
    if df.empty:
        return {'Scenario': name, 'Trades': 0}

    wins        = df['won'].sum()
    n           = len(df)
    total_gross = df['gross_pnl'].sum()
    total_fees  = df['fee_total'].sum()
    total_net   = df['net_pnl'].sum()
    avg_edge    = df['edge'].abs().mean()

    # Max drawdown on cumulative P&L
    cum = df['cumulative_pnl']
    roll_max = cum.cummax()
    drawdown  = (cum - roll_max).min()

    # Sharpe-like ratio (daily net P&L)
    daily_pnl = df.groupby('local_date')['net_pnl'].sum()
    sharpe = (daily_pnl.mean() / daily_pnl.std() * (252 ** 0.5)) if daily_pnl.std() > 0 else 0.0

    return {
        'Scenario':        name,
        'Trades':          n,
        'Win Rate':        f'{wins/n:.1%}',
        'Gross P&L':       f'${total_gross:>8.2f}',
        'Total Fees':      f'${total_fees:>7.2f}',
        'Net P&L':         f'${total_net:>8.2f}',
        'Avg |Edge|':      f'{avg_edge:.1%}',
        'Max Drawdown':    f'${drawdown:>7.2f}',
        'Ann. Sharpe':     f'{sharpe:>6.2f}',
    }


summary_rows = [scenario_summary(name, df) for name, df in backtest_results.items()]
summary_df   = pd.DataFrame(summary_rows)
print('\n' + '='*75)
print('BACKTEST SUMMARY — 5 Cities, Historical Data')
print('='*75)
print(f'  MIN_EDGE={MIN_EDGE:.0%}  MIN_MARKET_PROB={MIN_MARKET_PROB:.0%}  '
      f'CONTRACTS={CONTRACTS}  MAKER_FEE=${MAKER_FEE_PER_CONTRACT:.4f}/contract')
print('='*75)
display(summary_df.set_index('Scenario'))


## Cell 10 — Per-City Breakdown

In [ ]:
for scenario_name, bt in backtest_results.items():
    if bt.empty:
        continue
    print(f'\n── {scenario_name} ──')
    city_breakdown = (
        bt.groupby('city')
        .agg(
            trades       =('won',      'count'),
            wins         =('won',      'sum'),
            gross_pnl    =('gross_pnl','sum'),
            total_fees   =('fee_total','sum'),
            net_pnl      =('net_pnl',  'sum'),
            avg_edge     =('edge',     lambda x: x.abs().mean()),
        )
        .assign(
            win_rate   = lambda d: (d['wins']      / d['trades']).map('{:.1%}'.format),
            gross_pnl  = lambda d:  d['gross_pnl'] .map('${:>7.2f}'.format),
            total_fees = lambda d:  d['total_fees'].map('${:>6.2f}'.format),
            net_pnl    = lambda d:  d['net_pnl']   .map('${:>7.2f}'.format),
            avg_edge   = lambda d:  d['avg_edge']  .map('{:.1%}'.format),
        )
        .drop(columns=['wins'])
    )
    display(city_breakdown)


## Cell 11 — Cumulative P&L Curves (all 3 scenarios)

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(14, 5))

colors    = ['#2ecc71', '#f39c12', '#e74c3c']
linestyle = ['-', '--', ':']

for (name, bt), color, ls in zip(backtest_results.items(), colors, linestyle):
    if bt.empty:
        continue
    ax.plot(
        bt.index,
        bt['cumulative_pnl'],
        label=f'{name}  (net ${bt["net_pnl"].sum():.2f})',
        color=color,
        linewidth=2,
        linestyle=ls,
    )

ax.axhline(0, color='grey', linewidth=0.8, linestyle='--')
ax.set_title('Kalshi Weather Backtest — Cumulative Net P&L by Slippage Scenario', fontsize=13)
ax.set_xlabel('Trade #')
ax.set_ylabel('Cumulative Net P&L ($)')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(DATA_DIR / 'cumulative_pnl.png', dpi=150)
plt.show()
print('Plot saved to backtest_data/cumulative_pnl.png')


## Cell 12 — Detailed Trade Log (No Slippage scenario)

In [ ]:
bt_base = backtest_results.get('No Slippage', pd.DataFrame())

if not bt_base.empty:
    display_cols = [
        'city', 'local_date', 'hour', 'bracket_label', 'side',
        'model_prob', 'market_prob', 'edge',
        'predicted_high', 'actual_high',
        'entry_price', 'fee_total', 'gross_pnl', 'net_pnl', 'won',
    ]
    display_cols = [c for c in display_cols if c in bt_base.columns]

    pd.set_option('display.max_rows', 50)
    display(bt_base[display_cols].head(50))

    # Save full log
    bt_base.to_csv(DATA_DIR / 'trade_log_no_slippage.csv', index=False)
    print(f'\nFull trade log saved to backtest_data/trade_log_no_slippage.csv')
    print(f'Total trades: {len(bt_base)}')
